# Pivot and Reshape

The **same data** can be arranged in different **shapes**. Choosing the right shape is a skill: a model wants one shape, a chart another, a report a third.

- **Long format**: one row per observation (many rows, few columns). Best for `groupby`, plotting libraries (seaborn) and most ML pipelines.
- **Wide format**: one row per entity, one column per measurement (few rows, many columns). Best for reading, reports, correlation between columns, matrices.

Reshaping tools:

| Direction | Tools |
|---|---|
| long -> wide | `pivot`, `pivot_table`, `unstack` |
| wide -> long | `melt`, `stack`, `wide_to_long` |
| counts of combinations | `crosstab` |
| lists in a cell -> rows | `explode` |

**Contents**
1. Wide vs long (and tidy data)
2. Our data
3. `pivot`
4. `pivot_table`
5. `crosstab`
6. `melt`
7. `stack` and `unstack`
8. Working with the MultiIndex result
9. `wide_to_long`, `explode`, transpose
10. Time series: dates x entities
11. Mini AI/ML example: user-item matrix and a tiny recommender
12. Pandas 3.x notes
13. Common mistakes
14. Interview questions
15. Practice questions with solutions
16. Cheat sheet

In [1]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)

pandas: 3.0.6


---
# 1. Wide vs long (and tidy data)

Same information, two shapes:

```
LONG (one row per observation)          WIDE (one row per student)
student  subject  marks                 student   maths  science  english
Ravi     maths    85                    Ravi      85     78       80
Ravi     science  78                    Asha      92     95       88
Ravi     english  80
Asha     maths    92
...
```

**Tidy data** (the standard rule): each **variable** is a column, each **observation** is a row, each **type of observation** is a table. Long format is usually the "tidy" one.

| Need | Shape |
|---|---|
| `groupby`, filtering, most pandas operations | long |
| seaborn / plotly charts (`hue`, `facet`) | long |
| Training tables: features as columns, one row per sample | wide (features) |
| Correlation between columns, heatmaps, reports | wide |
| Matrix algorithms (recommenders, similarity) | wide |
| Storing events (logs, transactions) | long |

Reshaping is moving between the two.

---
# 2. Our data

Long sales table (12 rows) and a wide marks table.

> **Tip:** months sort alphabetically (`Feb`, `Jan`, `Mar`) in a pivot. We make `month` an **ordered categorical**, so the natural order is kept.

In [2]:
sales = pd.DataFrame({
    "month":   ["Jan", "Jan", "Jan", "Jan", "Feb", "Feb", "Feb", "Feb", "Mar", "Mar", "Mar", "Mar"],
    "region":  ["North", "North", "South", "South"] * 3,
    "product": ["Pen", "Book"] * 6,
    "units":   [10, 2, 8, 3, 12, 1, 9, 4, 15, 3, 7, 2],
    "revenue": [100, 500, 80, 750, 120, 250, 90, 1000, 150, 750, 70, 500],
})
sales["month"] = pd.Categorical(sales["month"], categories=["Jan", "Feb", "Mar"], ordered=True)
sales

,month,region,product,units,revenue
0,Jan,North,Pen,10,100
1,Jan,North,Book,2,500
2,Jan,South,Pen,8,80
3,Jan,South,Book,3,750
4,Feb,North,Pen,12,120
5,Feb,North,Book,1,250
6,Feb,South,Pen,9,90
7,Feb,South,Book,4,1000
8,Mar,North,Pen,15,150
9,Mar,North,Book,3,750


In [3]:
marks_wide = pd.DataFrame({
    "student": ["Ravi", "Asha", "Karan", "Neha"],
    "maths":   [85, 92, 58, 88],
    "science": [78, 95, 60, np.nan],
    "english": [80, 88, 70, 75],
})
marks_wide

,student,maths,science,english
0,Ravi,85,78.0,80
1,Asha,92,95.0,88
2,Karan,58,60.0,70
3,Neha,88,NaN,75


---
# 3. `pivot`: long to wide

```python
df.pivot(index="row_labels", columns="column_labels", values="cell_values")
```

- `index`: what becomes the **rows**
- `columns`: what becomes the **columns** (its unique values turn into column names)
- `values`: what fills the **cells**

**Rule:** each (index, columns) pair must be **unique**. `pivot` only rearranges, it does not aggregate.

In [4]:
# One row per (month, region, product) triple is unique if we use all three as the index/columns pair:
jan_north = sales[(sales["region"] == "North")]
jan_north.pivot(index="month", columns="product", values="revenue")

product,Book,Pen
month,,
Jan,500,100
Feb,250,120
Mar,750,150


In [5]:
# Several value columns at once
jan_north.pivot(index="month", columns="product", values=["units", "revenue"])

units     revenue     
product  Book Pen    Book  Pen
month                         
Jan         2  10     500  100
Feb         1  12     250  120
Mar         3  15     750  150

## The duplicate error

If you pivot `month` x `region` but each pair appears **twice** (once per product), `pivot` does not know which value to use:

In [6]:
try:
    sales.pivot(index="month", columns="region", values="revenue")
except ValueError as e:
    print("ValueError:", e)

ValueError: Index contains duplicate entries, cannot reshape


When pairs repeat, you must **aggregate**. That is what `pivot_table` does.

# 4. `pivot_table`: pivot + aggregation

Think of it as a **groupby with two keys, shown as a grid**.

```python
df.pivot_table(index=..., columns=..., values=..., aggfunc="mean", fill_value=None, margins=False)
```
Default `aggfunc` is **`"mean"`**, not sum. State it explicitly.

In [7]:
sales.pivot_table(index="month", columns="region", values="revenue", aggfunc="sum")

region,North,South
month,,
Jan,600,830
Feb,370,1090
Mar,900,570


In [8]:
# Same result with groupby + unstack (this is what happens inside)
sales.groupby(["month", "region"], observed=True)["revenue"].sum().unstack()

region,North,South
month,,
Jan,600,830
Feb,370,1090
Mar,900,570


## 4.1 Totals with `margins`

In [9]:
sales.pivot_table(index="month", columns="region", values="revenue", aggfunc="sum",
                  margins=True, margins_name="Total")

region,North,South,Total
month,,,
Jan,600,830,1430
Feb,370,1090,1460
Mar,900,570,1470
Total,1870,2490,4360


## 4.2 Several aggregations and several value columns

In [10]:
sales.pivot_table(index="region", columns="product", values="revenue", aggfunc=["sum", "mean"])

sum        mean            
product  Book  Pen   Book         Pen
region                               
North    1500  370  500.0  123.333333
South    2250  240  750.0   80.000000

In [11]:
# Different function per value column
sales.pivot_table(index="region", values=["units", "revenue"], aggfunc={"units": "sum", "revenue": "mean"})

,revenue,units
region,,
North,311.666667,43
South,415.000000,33


## 4.3 Several row/column keys

In [12]:
sales.pivot_table(index=["region", "product"], columns="month", values="revenue", aggfunc="sum")

month           Jan   Feb  Mar
region product                
North  Book     500   250  750
       Pen      100   120  150
South  Book     750  1000  500
       Pen       80    90   70

## 4.4 `fill_value`: empty cells

If a combination does not exist, the cell is NaN. For counts and sums, 0 is usually the honest value.

In [13]:
partial = sales[~((sales["region"] == "South") & (sales["month"] == "Mar"))]     # remove one combination

print(partial.pivot_table(index="region", columns="month", values="revenue", aggfunc="sum", observed=True))
print()
print(partial.pivot_table(index="region", columns="month", values="revenue", aggfunc="sum",
                          fill_value=0, observed=True))

month     Jan     Feb    Mar
region                      
North   600.0   370.0  900.0
South   830.0  1090.0    NaN

month   Jan   Feb  Mar
region                
North   600   370  900
South   830  1090    0


## 4.5 Percent of total (a very common request)

In [14]:
rev = sales.pivot_table(index="region", columns="product", values="revenue", aggfunc="sum")

share_of_row = rev.div(rev.sum(axis=1), axis=0).mul(100).round(1)        # each row adds to 100
share_of_all = (rev / rev.to_numpy().sum() * 100).round(1)                # whole table adds to 100

print("% within each region:")
print(share_of_row)
print()
print("% of total revenue:")
print(share_of_all)

% within each region:
product  Book   Pen
region             
North    80.2  19.8
South    90.4   9.6

% of total revenue:
product  Book  Pen
region            
North    34.4  8.5
South    51.6  5.5


`div(..., axis=0)` divides **row by row** (aligned on the index). Without `axis=0` pandas would align columns and give the wrong answer.

| | `pivot` | `pivot_table` |
|---|---|---|
| Aggregates duplicates | No (raises error) | **Yes** (`aggfunc`) |
| `fill_value`, `margins` | No | Yes |
| Use when | data is already unique per cell | typical analysis / reports |

---
# 5. `crosstab`: counting combinations

`pd.crosstab(rows, columns)` counts how often each **combination of categories** occurs. Perfect for relations between **two categorical columns**.

In [15]:
rng = np.random.default_rng(4)
n = 200

customers = pd.DataFrame({
    "plan": rng.choice(["Basic", "Standard", "Pro"], n, p=[0.5, 0.3, 0.2]),
})
churn_prob = customers["plan"].map({"Basic": 0.35, "Standard": 0.2, "Pro": 0.08})
customers["churned"] = np.where(rng.random(n) < churn_prob, "Yes", "No")

pd.crosstab(customers["plan"], customers["churned"])

churned,No,Yes
plan,,
Basic,54,34
Pro,44,3
Standard,56,9


In [16]:
# Add totals
pd.crosstab(customers["plan"], customers["churned"], margins=True)

churned,No,Yes,All
plan,,,
Basic,54,34,88
Pro,44,3,47
Standard,56,9,65
All,154,46,200


In [17]:
# normalize="index": each ROW adds up to 1 -> "of the customers in this plan, what share churned?"
(pd.crosstab(customers["plan"], customers["churned"], normalize="index") * 100).round(1)

churned,No,Yes
plan,,
Basic,61.4,38.6
Pro,93.6,6.4
Standard,86.2,13.8


In [18]:
# normalize="columns": each COLUMN adds up to 1 / normalize="all": the whole table adds up to 1
print((pd.crosstab(customers["plan"], customers["churned"], normalize="columns") * 100).round(1))
print()
print((pd.crosstab(customers["plan"], customers["churned"], normalize="all") * 100).round(1))

churned     No   Yes
plan                
Basic     35.1  73.9
Pro       28.6   6.5
Standard  36.4  19.6

churned     No   Yes
plan                
Basic     27.0  17.0
Pro       22.0   1.5
Standard  28.0   4.5


In [19]:
# crosstab can also aggregate a third column
pd.crosstab(sales["region"], sales["product"], values=sales["revenue"], aggfunc="sum")

product,Book,Pen
region,,
North,1500,370
South,2250,240


In ML, a crosstab with `normalize="index"` is the quickest way to see **how a category relates to the target** (here: plan vs churn). The Basic plan churns far more, so `plan` is a useful feature.

| | `groupby().size().unstack()` | `crosstab` | `pivot_table(aggfunc="count")` |
|---|---|---|---|
| Counts combinations | yes | **yes, shortest** | yes |
| Normalize built in | no | **yes** | no |
| Works on DataFrame columns | yes | takes Series | yes |

---
# 6. `melt`: wide to long

```python
df.melt(id_vars=[...columns to keep...], value_vars=[...columns to unpivot...], var_name="...", value_name="...")
```

In [20]:
marks_long = marks_wide.melt(id_vars="student", var_name="subject", value_name="marks")
marks_long

,student,subject,marks
0,Ravi,maths,85.0
1,Asha,maths,92.0
2,Karan,maths,58.0
3,Neha,maths,88.0
4,Ravi,science,78.0
5,Asha,science,95.0
6,Karan,science,60.0
7,Neha,science,NaN
8,Ravi,english,80.0
9,Asha,english,88.0


In [21]:
# value_vars chooses which columns to melt; the rest of the columns need to be in id_vars
marks_wide.melt(id_vars="student", value_vars=["maths", "english"], var_name="subject", value_name="marks")

,student,subject,marks
0,Ravi,maths,85
1,Asha,maths,92
2,Karan,maths,58
3,Neha,maths,88
4,Ravi,english,80
5,Asha,english,88
6,Karan,english,70
7,Neha,english,75


In [22]:
# Long format makes group analysis easy
print(marks_long.groupby("subject")["marks"].mean().round(1))
print()
print(marks_long.groupby("student")["marks"].agg(["mean", "max"]).round(1))

subject
english    78.2
maths      80.8
science    77.7
Name: marks, dtype: float64

         mean   max
student            
Asha     91.7  95.0
Karan    62.7  70.0
Neha     81.5  88.0
Ravi     81.0  85.0


In [23]:
# Best subject of each student (NaN is skipped)
best = marks_long.loc[marks_long.groupby("student")["marks"].idxmax()]
best[["student", "subject", "marks"]]

,student,subject,marks
5,Asha,science,95.0
10,Karan,english,70.0
3,Neha,maths,88.0
0,Ravi,maths,85.0


**Why melt for ML and charts?** Imagine 12 monthly columns (`jan`, `feb`, ... `dec`). In long format they become a single `month` column that you can group, filter, encode, or use as the colour in a chart (`sns.lineplot(data=long, x="month", y="value", hue="store")`).

Melt and pivot are **inverses**:

In [24]:
back = marks_long.pivot(index="student", columns="subject", values="marks").reset_index()
back.columns.name = None
back[["student", "maths", "science", "english"]]

,student,maths,science,english
0,Asha,92.0,95.0,88.0
1,Karan,58.0,60.0,70.0
2,Neha,88.0,NaN,75.0
3,Ravi,85.0,78.0,80.0


---
# 7. `stack` and `unstack`

They move a **level of the index** between rows and columns. They work best with a **MultiIndex**.

- `unstack()`: innermost **row** level -> **columns** (long -> wide)
- `stack()`: **columns** -> innermost **row** level (wide -> long)

In [25]:
by_two = sales.groupby(["region", "product"])["revenue"].sum()
print(by_two)

region  product
North   Book       1500
        Pen         370
South   Book       2250
        Pen         240
Name: revenue, dtype: int64


In [26]:
by_two.unstack()                          # product moves to the columns

product,Book,Pen
region,,
North,1500,370
South,2250,240


In [27]:
by_two.unstack(level="region")            # choose which level moves

region,North,South
product,,
Book,1500,2250
Pen,370,240


In [28]:
by_two.unstack(fill_value=0)              # fill gaps

product,Book,Pen
region,,
North,1500,370
South,2250,240


In [29]:
# stack(): the reverse
wide = by_two.unstack()
wide.stack()

region  product
North   Book       1500
        Pen         370
South   Book       2250
        Pen         240
dtype: int64

**Pandas 3 note:** the new `stack()` **keeps NaN cells** as rows (it does not drop them as the old version did) and the `dropna` argument is no longer allowed. If you want them gone, call `.dropna()` afterwards.

In [30]:
nums = pd.DataFrame({"a": [1, np.nan], "b": [3, 4]}, index=["x", "y"])
stacked = nums.stack()
print(stacked)
print("NaN removed:", stacked.dropna().shape[0], "rows")

x  a    1.0
   b    3.0
y  a    NaN
   b    4.0
dtype: float64
NaN removed: 3 rows


In [31]:
# stack on marks: wide -> long (similar to melt, but the id goes into the index)
marks_wide.set_index("student").stack().rename("marks").reset_index().rename(columns={"level_1": "subject"}).head(4)

,student,subject,marks
0,Ravi,maths,85.0
1,Ravi,science,78.0
2,Ravi,english,80.0
3,Asha,maths,92.0


| | `melt` | `stack` |
|---|---|---|
| Works on | plain columns (`id_vars`) | the column labels, index stays as id |
| Names the new columns | directly (`var_name`, `value_name`) | needs rename afterwards |
| Typical use | tidy up a wide table | after groupby / pivot with a MultiIndex |

| | `pivot` | `unstack` |
|---|---|---|
| Works on | plain columns | an **index level** |
| Aggregates duplicates | no | no (needs a unique index) |

---
# 8. Working with the MultiIndex result

Pivot tables with several keys have **MultiIndex** rows or columns.

In [32]:
pt = sales.pivot_table(index=["region", "product"], columns="month", values="revenue", aggfunc="sum", observed=True)
pt

month           Jan   Feb  Mar
region product                
North  Book     500   250  750
       Pen      100   120  150
South  Book     750  1000  500
       Pen       80    90   70

In [33]:
pt.loc["North"]                                 # one outer label -> sub-table

month,Jan,Feb,Mar
product,,,
Book,500,250,750
Pen,100,120,150


In [34]:
pt.loc[("North", "Book")]                       # a full key -> Series

month
Jan    500
Feb    250
Mar    750
Name: (North, Book), dtype: int64

In [35]:
pt.xs("Pen", level="product")                   # "cross-section": select on an inner level

month,Jan,Feb,Mar
region,,,
North,100,120,150
South,80,90,70


In [36]:
pt.reset_index()                                # flatten the row MultiIndex into normal columns

month,region,product,Jan,Feb,Mar
0,North,Book,500,250,750
1,North,Pen,100,120,150
2,South,Book,750,1000,500
3,South,Pen,80,90,70


In [37]:
# MultiIndex COLUMNS (from several aggfuncs) -> flatten the names
multi = sales.pivot_table(index="region", columns="product", values="revenue", aggfunc=["sum", "mean"])
multi.columns = ["_".join(col) for col in multi.columns]
multi

,sum_Book,sum_Pen,mean_Book,mean_Pen
region,,,,
North,1500,370,500.0,123.333333
South,2250,240,750.0,80.000000


In [38]:
# Columns name left over from a pivot ("product") can be removed
flat = sales.pivot_table(index="region", columns="product", values="revenue", aggfunc="sum")
flat.columns.name = None
flat = flat.reset_index()
flat

,region,Book,Pen
0,North,1500,370
1,South,2250,240


Handy tools: `droplevel(level)`, `swaplevel()`, `reorder_levels([...])`, `sort_index()`.

**Tip:** before saving to CSV or feeding a model, **flatten**: `reset_index()` plus flat column names.

---
# 9. `wide_to_long`, `explode`, transpose

## 9.1 `wide_to_long`: columns with a common prefix and a suffix (like a year)

In [39]:
rev_wide = pd.DataFrame({
    "dept": ["ML", "Web", "Data"],
    "rev_2024": [100, 80, 60],
    "rev_2025": [120, 90, 65],
    "cost_2024": [70, 50, 40],
    "cost_2025": [80, 55, 45],
})

pd.wide_to_long(rev_wide, stubnames=["rev", "cost"], i="dept", j="year", sep="_").reset_index()

,dept,year,rev,cost
0,ML,2024,100,70
1,Web,2024,80,50
2,Data,2024,60,40
3,ML,2025,120,80
4,Web,2025,90,55
5,Data,2025,65,45


Two measurements (`rev`, `cost`) stay as columns, the year becomes one column. With plain `melt` you would need extra string splitting.

## 9.2 `explode`: a list inside a cell -> one row per item

In [40]:
people = pd.DataFrame({
    "name": ["Ravi", "Asha", "Karan"],
    "skills": [["python", "sql"], ["python", "docker", "aws"], ["sql"]],
})

skills_long = people.explode("skills")
print(skills_long.index.tolist())          # the original index is repeated: 0, 0, 1, 1, 1, 2
skills_long

[0, 0, 1, 1, 1, 2]


,name,skills
0,Ravi,python
0,Ravi,sql
1,Asha,python
1,Asha,docker
1,Asha,aws
2,Karan,sql


`explode` keeps the original index (so it repeats). This can cause
problems when combining it with other objects that require unique or
compatible index alignment. When you do not need the old index, use
`ignore_index=True`.

In [41]:
skills_long = people.explode("skills", ignore_index=True)
skills_long

,name,skills
0,Ravi,python
1,Ravi,sql
2,Asha,python
3,Asha,docker
4,Asha,aws
5,Karan,sql


In [42]:
print(skills_long["skills"].value_counts())
print()
# back: one row per person with a list again
skills_long.groupby("name")["skills"].agg(list)

skills
python    2
sql       2
docker    1
aws       1
Name: count, dtype: int64



name
Asha     [python, docker, aws]
Karan                    [sql]
Ravi             [python, sql]
Name: skills, dtype: object

In [43]:
# Text with separators: split first, then explode
tags = pd.DataFrame({"post": [1, 2], "tags": ["ml,python", "sql"]})
tags.assign(tags=tags["tags"].str.split(",")).explode("tags")

,post,tags
0,1,ml
0,1,python
1,2,sql


In [44]:
# Multi-hot (one 0/1 column per skill) from the exploded table
pd.crosstab(skills_long["name"], skills_long["skills"])

skills,aws,docker,python,sql
name,,,,
Asha,1,1,1,0
Karan,0,0,0,1
Ravi,0,0,1,1


## 9.3 Transpose `.T`

Swaps rows and columns. Fine for **small** summary tables (like `describe()`); avoid it on large mixed-type data.

In [45]:
marks_wide.set_index("student").describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
maths,4.0,80.8,15.4,58.0,78.2,86.5,89.0,92.0
science,3.0,77.7,17.5,60.0,69.0,78.0,86.5,95.0
english,4.0,78.2,7.7,70.0,73.8,77.5,82.0,88.0


---
# 10. Time series: dates x entities

Daily units for three stores in **long** format (how logs arrive), then reshaped for different jobs.

In [46]:
rng = np.random.default_rng(8)
days = pd.date_range("2026-03-01", periods=10)

ts_long = pd.DataFrame({
    "date": np.repeat(days, 3),
    "store": ["A", "B", "C"] * 10,
    "units": rng.integers(20, 60, 30),
})
ts_long.head(6)

,date,store,units
0,2026-03-01,A,48
1,2026-03-01,B,33
2,2026-03-01,C,29
3,2026-03-02,A,59
4,2026-03-02,B,27
5,2026-03-02,C,32


In [47]:
# long -> wide: one column per store
ts_wide = ts_long.pivot(index="date", columns="store", values="units")
ts_wide

store,A,B,C
date,,,
2026-03-01,48,33,29
2026-03-02,59,27,32
2026-03-03,45,51,45
2026-03-04,54,21,35
2026-03-05,42,37,35
2026-03-06,34,21,24
2026-03-07,41,39,58
2026-03-08,29,53,30
2026-03-09,25,27,35


In [48]:
# Wide format is convenient for column-wise comparisons
print(ts_wide.corr().round(2))                # how similar are the stores?
print()
print(ts_wide.sum(axis=1).head(3))            # total across stores per day

store     A     B     C
store                  
A      1.00 -0.40 -0.13
B     -0.40  1.00  0.52
C     -0.13  0.52  1.00

date
2026-03-01    110
2026-03-02    118
2026-03-03    141
dtype: int64


In [49]:
# A weekly table: week x store
ts_long.pivot_table(index=pd.Grouper(key="date", freq="W"), columns="store", values="units", aggfunc="sum")

store,A,B,C
date,,,
2026-03-01,48,33,29
2026-03-08,304,249,259
2026-03-15,52,82,87


In [50]:
# Features on the wide table are easy (all stores at once), then back to long for modelling
feat = pd.concat({
    "units": ts_wide,
    "lag_1": ts_wide.shift(1),
    "roll3": ts_wide.shift(1).rolling(3).mean().round(1),      # shift first: no leakage (chapter 09)
}, axis=1)

feat.stack(level=1).dropna().reset_index().head(6)

,date,store,units,lag_1,roll3
0,2026-03-04,A,54,45.0,50.7
1,2026-03-04,B,21,51.0,37.0
2,2026-03-04,C,35,45.0,35.3
3,2026-03-05,A,42,54.0,52.7
4,2026-03-05,B,37,21.0,33.0
5,2026-03-05,C,35,35.0,37.3


In [51]:
# Back to long (e.g. for seaborn: sns.lineplot(data=long, x="date", y="units", hue="store"))
ts_wide.reset_index().melt(id_vars="date", var_name="store", value_name="units").head(4)

,date,store,units
0,2026-03-01,A,48
1,2026-03-02,A,59
2,2026-03-03,A,45
3,2026-03-04,A,54


---
# 11. Mini AI/ML example: user-item matrix and a tiny recommender

**Recommender systems** start with a **user x item matrix**: rows = users, columns = items, cells = ratings. Raw data arrives in **long** format (one row per rating), so the first step is a `pivot`.

In [52]:
ratings = pd.DataFrame({
    "user":   ["Ravi", "Ravi", "Ravi", "Asha", "Asha", "Asha", "Asha", "Karan", "Karan", "Karan",
               "Neha", "Neha", "Neha", "Amit", "Amit", "Amit", "Amit"],
    "movie":  ["Dhoom", "Lagaan", "3 Idiots", "Dhoom", "Lagaan", "Dangal", "3 Idiots", "Lagaan", "Dangal", "Swades",
               "Dhoom", "3 Idiots", "Swades", "Lagaan", "Dangal", "Swades", "3 Idiots"],
    "rating": [5, 3, 4, 4, 2, 3, 5, 4, 5, 4, 5, 5, 2, 3, 4, 5, 4],
})

matrix = ratings.pivot(index="user", columns="movie", values="rating")
matrix

movie,3 Idiots,Dangal,Dhoom,Lagaan,Swades
user,,,,,
Amit,4.0,4.0,NaN,3.0,5.0
Asha,5.0,3.0,4.0,2.0,NaN
Karan,NaN,5.0,NaN,4.0,4.0
Neha,5.0,NaN,5.0,NaN,2.0
Ravi,4.0,NaN,5.0,3.0,NaN


In [53]:
# Sparsity: how much of the matrix is empty? (real recommenders: 95-99% empty)
filled = matrix.notna().to_numpy().mean()
print(f"filled cells: {filled:.0%} | empty: {1 - filled:.0%}")

filled cells: 68% | empty: 32%


## Item-item similarity

Two movies are **similar** if the same users rate them in the same way. Steps:
1. Subtract each user's average rating (some users rate everything high)
2. Fill empty cells with 0 (no information)
3. Compute **cosine similarity** between the movie columns

In [54]:
centered = matrix.sub(matrix.mean(axis=1), axis=0).fillna(0)       # step 1 and 2

M = centered.to_numpy()
norms = np.linalg.norm(M, axis=0)
sim = pd.DataFrame((M.T @ M) / np.outer(norms, norms), index=matrix.columns, columns=matrix.columns)

sim.round(2)

movie,3 Idiots,Dangal,Dhoom,Lagaan,Swades
movie,,,,,
3 Idiots,1.00,-0.50,0.65,-0.60,-0.49
Dangal,-0.50,1.00,-0.20,0.30,-0.12
Dhoom,0.65,-0.20,1.00,-0.56,-0.59
Lagaan,-0.60,0.30,-0.56,1.00,-0.19
Swades,-0.49,-0.12,-0.59,-0.19,1.00


In [55]:
# Most similar movie to each movie
(
    sim.where(~np.eye(len(sim), dtype=bool))      # hide the diagonal (a movie is 100% similar to itself)
       .idxmax()
       .rename("most_similar_to_it")
)

movie
3 Idiots       Dhoom
Dangal        Lagaan
Dhoom       3 Idiots
Lagaan        Dangal
Swades        Dangal
Name: most_similar_to_it, dtype: str

In [56]:
def recommend(user, top_n=2):
    seen = matrix.loc[user].dropna()                 # movies the user rated
    user_avg = seen.mean()
    unseen = [m for m in matrix.columns if m not in seen.index]

    scores = {}
    for movie in unseen:
        weights = sim.loc[movie, seen.index]
        # start from the user's own average, move up/down by similarity-weighted deviations
        scores[movie] = user_avg + (weights * (seen - user_avg)).sum() / weights.abs().sum()

    return pd.Series(scores, name="predicted_rating").sort_values(ascending=False).head(top_n).round(2)

print("Ravi has seen:", matrix.loc["Ravi"].dropna().index.tolist())
recommend("Ravi")

Ravi has seen: ['3 Idiots', 'Dhoom', 'Lagaan']


Swades    3.68
Dangal    3.50
Name: predicted_rating, dtype: float64

The prediction starts from the user's **own average rating** and moves up or down depending on how similar the unseen movie is to the movies the user liked or disliked. (With only 5 users the similarities are noisy.)

This is a **toy** version (real systems use millions of ratings, sparse matrices and matrix factorization), but the pipeline is exactly this:

1. **long** event table (`user`, `item`, `rating`)
2. **`pivot`** -> user x item matrix
3. similarity or factorization on the matrix
4. **back to long** (`melt` / `stack`) to store recommendations or train a model

Other places where `pivot` / `crosstab` build ML features:
- **Category spending per customer** (customer x category matrix of amounts, chapter 11)
- **Bag of events** (user x event_type counts) for churn models
- **Multi-hot encoding** of lists (`explode` + `crosstab`)

In [57]:
# Features from a long event log with pivot_table (customer x event_type counts)
events = pd.DataFrame({
    "customer": ["c1", "c1", "c1", "c2", "c2", "c3", "c3", "c3", "c3"],
    "event":    ["login", "login", "purchase", "login", "support_call", "login", "purchase", "purchase", "support_call"],
})

event_features = events.pivot_table(index="customer", columns="event", aggfunc="size", fill_value=0).add_prefix("n_")
event_features.columns.name = None
event_features

,n_login,n_purchase,n_support_call
customer,,,
c1,2,1,0
c2,1,0,1
c3,1,2,1


---
# 12. Pandas 3.x notes

- **`stack()`** uses the new implementation: NaN cells are **kept**, `dropna=` is not allowed. Call `.dropna()` yourself.
- `pivot_table` default `aggfunc` is `"mean"`. Always pass `aggfunc` explicitly.
- `pivot_table` / `groupby` with categorical keys use **`observed=True`** by default, so empty categories do not appear. Add `observed=False` to show them.
- A pivot result has a **column index name** (like `product`) that shows up as a header. Remove with `df.columns.name = None`.
- `pivot` raises `ValueError` for duplicate (index, columns) pairs. Use `pivot_table`.
- Text columns use the `str` dtype. Pivoted column labels from text become `str` labels.

---
# 13. Common mistakes

| Mistake | Fix |
|---|---|
| `pivot` fails with "duplicate entries" | `pivot_table` with an `aggfunc` (or remove the duplicates on purpose) |
| `pivot_table` without `aggfunc` and surprise averages | `aggfunc="sum"` (default is mean) |
| Months in alphabetical order (Apr, Aug, Dec...) | Ordered `Categorical`, or sort with a month-number column |
| Empty cells stay NaN in count/sum tables | `fill_value=0` |
| `df.div(df.sum(axis=1))` gives wrong numbers | `df.div(df.sum(axis=1), axis=0)` |
| Keeping MultiIndex columns/rows in a model or CSV | Flatten: `reset_index()`, join column names |
| Forgetting `id_vars` in `melt` (all columns melted) | Put identifying columns into `id_vars` |
| Using `stack().dropna=True` (old syntax) | `.stack().dropna()` |
| Pivoting very large tables to wide with thousands of columns | Keep long, or use sparse matrices |
| Filling the user-item matrix with 0 and treating it as "dislike" | 0 means "unknown". Center ratings first, or use sparse methods |
| Wide table for a chart library that expects long data | `melt` first |
| `crosstab` counts instead of percentages when groups differ in size | `normalize="index"` |
| Lists in cells treated as normal values | `explode` them (`ignore_index=True` if you will use `crosstab`/alignment next) |

---
# 14. Interview questions

**Q1. What is the difference between wide and long format?**
Wide: one row per entity, one column per variable/time. Long: one row per observation with variable names in a column. Long is the "tidy" shape and works best with `groupby` and plotting libraries; wide is better for reports, matrices and correlations.

**Q2. `pivot` vs `pivot_table`?**
`pivot` only reshapes and fails when (index, columns) pairs repeat. `pivot_table` also aggregates duplicates with `aggfunc` (default mean) and supports `fill_value` and `margins`.

**Q3. What does `melt` do?**
Turns columns into rows: `id_vars` stay as identifiers, the other columns become two columns (variable name and value). It is the inverse of `pivot`.

**Q4. `stack` vs `unstack`?**
`stack` moves column labels into the row index (wide to long). `unstack` moves an index level into columns (long to wide).

**Q5. What is `crosstab` and when do you use it?**
It counts the combinations of two (or more) categorical columns, with `normalize` for percentages. It is used to see how a category relates to another one, such as plan vs churn.

**Q6. How do you get percent of row total from a pivot table?**
`pt.div(pt.sum(axis=1), axis=0) * 100`.

**Q7. How do you flatten a MultiIndex after `pivot_table`?**
`df.columns = ["_".join(c) for c in df.columns]` for MultiIndex columns and `reset_index()` for the row index.

**Q8. How would you turn a column of lists into rows?**
`df.explode("col")`, and back with `groupby(...).agg(list)`.

**Q9. Why are user-item matrices a good example of pivoting?**
Ratings arrive as long events (user, item, rating). Pivoting gives a matrix with users as rows and items as columns, which similarity and factorization methods need.

**Q10. When would you keep data in long format for ML?**
For time series with many entities (features per entity via groupby/shift), for models with an entity ID as a categorical feature, and whenever you use `groupby` or plotting libraries.

**Q11. What does `wide_to_long` do and why is it handy?**
Converts columns like `rev_2024`, `rev_2025` into rows with a `year` column while keeping several measurements as columns. Easier than melt plus string splitting.

**Q12. How can the order of months in a pivot table be fixed?**
Convert the month column to an ordered `Categorical` with the natural order, or sort by a numeric month column.

---
# 15. Practice questions

Run this cell first to get the practice data.

In [58]:
sp = pd.DataFrame({
    "salesperson": ["Ravi", "Ravi", "Asha", "Asha", "Ravi", "Asha", "Karan", "Karan", "Ravi", "Asha", "Karan", "Karan"],
    "quarter":     ["Q1", "Q1", "Q1", "Q2", "Q2", "Q2", "Q1", "Q2", "Q3", "Q3", "Q3", "Q4"],
    "product":     ["Pen", "Book", "Pen", "Book", "Pen", "Pen", "Book", "Book", "Bag", "Bag", "Pen", "Book"],
    "revenue":     [100, 500, 120, 600, 150, 130, 550, 480, 800, 900, 140, 520],
})

marks_p = pd.DataFrame({
    "student": ["Ravi", "Asha", "Karan", "Neha"],
    "maths":   [85, 92, 58, 88],
    "science": [78, 95, 60, np.nan],
    "english": [80, 88, 70, 75],
})

rev_p = pd.DataFrame({"dept": ["ML", "Web", "Data"], "rev_2024": [100, 80, 60], "rev_2025": [120, 90, 65]})

skills_p = pd.DataFrame({
    "name": ["Ravi", "Asha", "Karan"],
    "skills": [["python", "sql"], ["python", "docker", "aws"], ["sql"]],
})
sp

,salesperson,quarter,product,revenue
0,Ravi,Q1,Pen,100
1,Ravi,Q1,Book,500
2,Asha,Q1,Pen,120
3,Asha,Q2,Book,600
4,Ravi,Q2,Pen,150
5,Asha,Q2,Pen,130
6,Karan,Q1,Book,550
7,Karan,Q2,Book,480
8,Ravi,Q3,Bag,800
9,Asha,Q3,Bag,900


**Q1.** Pivot total revenue: rows = `salesperson`, columns = `quarter`.
**Q2.** Same pivot with `fill_value=0` and a grand total row and column named `Total`.
**Q3.** A pivot with `sum` **and** `count` of revenue per `salesperson` and `product`. Flatten the column names.
**Q4.** Crosstab of `product` by `quarter` (counts), and the same normalized so each **row** adds to 100%.
**Q5.** Melt `marks_p` into long format (`student`, `subject`, `marks`), dropping the missing mark.
**Q6.** From the long marks table: average marks per subject, and the best subject per student.
**Q7.** `sp.groupby(["salesperson", "product"])["revenue"].sum()`: unstack `product` with 0 for gaps, then stack back and drop zeros.
**Q8.** Use `wide_to_long` on `rev_p` to get `dept`, `year`, `rev`.
**Q9.** Explode `skills_p`, count how many people know each skill, and build a multi-hot table (person x skill).
**Q10.** Show that `sp.pivot(index="salesperson", columns="quarter", values="revenue")` fails, and explain why.
**Q11.** From the `ratings` table of section 11, find the **most similar pair of movies** using `pivot` and `corr()` (pairwise correlation of the movie columns). Ignore pairs rated by fewer than 3 common users (`min_periods=3`). Even 3 is very few: real systems need many more.
**Q12.** Using the pivot from Q1, compute each salesperson's share (%) of every quarter's revenue (each **column** adds to 100).

### Solutions

In [59]:
# Q1
sp.pivot_table(index="salesperson", columns="quarter", values="revenue", aggfunc="sum")

quarter,Q1,Q2,Q3,Q4
salesperson,,,,
Asha,120.0,730.0,900.0,NaN
Karan,550.0,480.0,140.0,520.0
Ravi,600.0,150.0,800.0,NaN


In [60]:
# Q2
sp.pivot_table(index="salesperson", columns="quarter", values="revenue", aggfunc="sum",
               fill_value=0, margins=True, margins_name="Total")

quarter,Q1,Q2,Q3,Q4,Total
salesperson,,,,,
Asha,120,730,900,0,1750
Karan,550,480,140,520,1690
Ravi,600,150,800,0,1550
Total,1270,1360,1840,520,4990


In [61]:
# Q3
q3 = sp.pivot_table(index="salesperson", columns="product", values="revenue", aggfunc=["sum", "count"], fill_value=0)
q3.columns = ["_".join(c) for c in q3.columns]
q3

,sum_Bag,sum_Book,sum_Pen,count_Bag,count_Book,count_Pen
salesperson,,,,,,
Asha,900,600,250,1,1,2
Karan,0,1550,140,0,3,1
Ravi,800,500,250,1,1,2


In [62]:
# Q4
print(pd.crosstab(sp["product"], sp["quarter"]))
print()
(pd.crosstab(sp["product"], sp["quarter"], normalize="index") * 100).round(1)

quarter  Q1  Q2  Q3  Q4
product                
Bag       0   0   2   0
Book      2   2   0   1
Pen       2   2   1   0



quarter,Q1,Q2,Q3,Q4
product,,,,
Bag,0.0,0.0,100.0,0.0
Book,40.0,40.0,0.0,20.0
Pen,40.0,40.0,20.0,0.0


In [63]:
# Q5
q5 = marks_p.melt(id_vars="student", var_name="subject", value_name="marks").dropna(subset=["marks"])
q5

,student,subject,marks
0,Ravi,maths,85.0
1,Asha,maths,92.0
2,Karan,maths,58.0
3,Neha,maths,88.0
4,Ravi,science,78.0
5,Asha,science,95.0
6,Karan,science,60.0
8,Ravi,english,80.0
9,Asha,english,88.0
10,Karan,english,70.0


In [64]:
# Q6
print(q5.groupby("subject")["marks"].mean().round(1))
print()
q5.loc[q5.groupby("student")["marks"].idxmax(), ["student", "subject", "marks"]]

subject
english    78.2
maths      80.8
science    77.7
Name: marks, dtype: float64



,student,subject,marks
5,Asha,science,95.0
10,Karan,english,70.0
3,Neha,maths,88.0
0,Ravi,maths,85.0


In [65]:
# Q7
g = sp.groupby(["salesperson", "product"])["revenue"].sum()
wide = g.unstack(fill_value=0)
print(wide)
print()
back = wide.stack()
back[back != 0]

product      Bag  Book  Pen
salesperson                
Asha         900   600  250
Karan          0  1550  140
Ravi         800   500  250



salesperson  product
Asha         Bag         900
             Book        600
             Pen         250
Karan        Book       1550
             Pen         140
Ravi         Bag         800
             Book        500
             Pen         250
dtype: int64

In [66]:
# Q8
pd.wide_to_long(rev_p, stubnames="rev", i="dept", j="year", sep="_").reset_index()

,dept,year,rev
0,ML,2024,100
1,Web,2024,80
2,Data,2024,60
3,ML,2025,120
4,Web,2025,90
5,Data,2025,65


In [67]:
# Q9
sk = skills_p.explode("skills", ignore_index=True)
print(sk.groupby("skills")["name"].nunique())
print()
pd.crosstab(sk["name"], sk["skills"])

skills
aws       1
docker    1
python    2
sql       2
Name: name, dtype: int64



skills,aws,docker,python,sql
name,,,,
Asha,1,1,1,0
Karan,0,0,0,1
Ravi,0,0,1,1


In [68]:
# Q10
try:
    sp.pivot(index="salesperson", columns="quarter", values="revenue")
except ValueError as e:
    print("ValueError:", e)
# Ravi has two rows in Q1 (Pen and Book): the pair (Ravi, Q1) is not unique, so pivot cannot choose.

ValueError: Index contains duplicate entries, cannot reshape


In [69]:
# Q11
# Without min_periods, pairs rated by only 2 users get a "perfect" correlation of 1.0 by chance
print("pairs with 1.0 and no min_periods:", int((matrix.corr().where(~np.eye(5, dtype=bool)) == 1.0).sum().sum() // 2))

pairs = matrix.corr(min_periods=3)                           # NaN if fewer than 3 common users
pairs = pairs.where(np.triu(np.ones(pairs.shape, dtype=bool), k=1))       # keep each pair once
best = pairs.stack().sort_values(ascending=False)           # pandas 3: stack keeps NaN, so drop them
print(best.dropna().head(3).round(2))
# Dangal-Lagaan is perfectly correlated, but it rests on only 3 shared users, so it is weak evidence.

pairs with 1.0 and no min_periods: 2
movie     movie 
Dangal    Lagaan    1.0
3 Idiots  Dhoom    -0.5
          Lagaan   -1.0
dtype: float64


In [70]:
# Q12
p = sp.pivot_table(index="salesperson", columns="quarter", values="revenue", aggfunc="sum", fill_value=0)
(p.div(p.sum(axis=0), axis=1) * 100).round(1)

quarter,Q1,Q2,Q3,Q4
salesperson,,,,
Asha,9.4,53.7,48.9,0.0
Karan,43.3,35.3,7.6,100.0
Ravi,47.2,11.0,43.5,0.0


---
# 16. Cheat sheet

```python
# LONG -> WIDE
df.pivot(index="r", columns="c", values="v")                         # unique pairs only
df.pivot_table(index="r", columns="c", values="v", aggfunc="sum",     # aggregates duplicates
               fill_value=0, margins=True, margins_name="Total")
df.pivot_table(index=["a", "b"], columns="c", values="v", aggfunc=["sum", "mean"])
df.groupby(["r", "c"])["v"].sum().unstack(fill_value=0)

# COUNTS OF COMBINATIONS
pd.crosstab(df["a"], df["b"])
pd.crosstab(df["a"], df["b"], normalize="index")                      # row %  (index | columns | all)
pd.crosstab(df["a"], df["b"], values=df["v"], aggfunc="sum", margins=True)

# WIDE -> LONG
df.melt(id_vars=["id"], value_vars=["x", "y"], var_name="var", value_name="val")
pd.wide_to_long(df, stubnames=["rev", "cost"], i="id", j="year", sep="_")
df.set_index("id").stack().dropna()                                   # pandas 3: NaN is kept, drop it yourself
df.explode("list_col")

# PERCENT OF TOTAL
pt.div(pt.sum(axis=1), axis=0)          # share within each row
pt.div(pt.sum(axis=0), axis=1)          # share within each column
pt / pt.to_numpy().sum()                # share of the grand total

# MULTIINDEX
pt.reset_index()     pt.loc["A"]     pt.loc[("A", "x")]     pt.xs("x", level="b")
pt.columns = ["_".join(c) for c in pt.columns]      pt.columns.name = None
pt.droplevel(0, axis=1)     pt.swaplevel()

# ORDER OF CATEGORIES
df["month"] = pd.Categorical(df["month"], categories=["Jan", "Feb", "Mar"], ordered=True)

# ML PATTERNS
matrix = ratings.pivot(index="user", columns="item", values="rating")           # user-item matrix
feats = events.pivot_table(index="id", columns="event", aggfunc="size", fill_value=0)   # event counts
pd.crosstab(df["category"], df["target"], normalize="index")                    # category vs target
```

**Next chapter:** `13_datetime_timeseries`: the `.dt` accessor, parsing dates, time zones, `resample`, `rolling` windows by time, date features for ML (day of week, month, holiday flags, cyclical encoding), and time-based train/test splits.